In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp
import json


# ============================================================
# 1. Read Kafka configuration from Databricks Secret Scope
# ============================================================

kafka_connection_json = dbutils.secrets.get(
    scope="finguard-scope",
    key="kafka_connection_details"
)

kafka_config = json.loads(kafka_connection_json)

bootstrap_servers = kafka_config["bootstrap_servers"]
api_key = kafka_config["api_key"]
api_secret = kafka_config["api_secret"]
topic = kafka_config["topic"]


# ============================================================
# 2. Kafka authentication configuration
# ============================================================

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
    f'required username="{api_key}" password="{api_secret}";'
)


# ============================================================
# 3. Build the Kafka source stream at module scope
#    NOTE: the @dp.table function below is only executed by a
#    pipeline, never when this notebook runs interactively, so
#    parsed_stream must be built here for the writeStream in the
#    next cell to reference it.
# ============================================================

stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("startingOffsets", "earliest")
    .load()
)

parsed_stream = stream_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    col("topic"),
    col("offset"),
    col("timestamp"),
    col("partition"),
    col("timestampType")
)


# ============================================================
# 4. Bronze streaming table (used when this code runs in a pipeline)
# ============================================================

@dp.table(
    name="finguard.bronze.transactions",
    comment="Raw transaction events ingested from Kafka"
)
def transactions():
    return parsed_stream

In [0]:

# NOTE: writes to a separate TEST table. finguard.bronze.transactions is a
# streaming table managed by a pipeline; writing to it from this notebook
# is not allowed and would duplicate the pipeline's own ingestion of the
# same Kafka topic.
streaming_query=(parsed_stream.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint_location/")
.trigger(availableNow=True)
.toTable("finguard.bronze.transactions_kafka_test")
)